[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HSajeed/silent-field/blob/tier0-review-notebooks/notebooks/05_review_findings.ipynb)

# 05 — Review findings (2026-10-01)

**What you will learn:** what the post-Tier-0 review (`review_2026-10-01.md`) found, with
every computed claim recomputed here and every cited claim quoted with its source.
1. The far field (and R_S) is built on a source form that is wrong for a periodic box.
2. The Stage-4 conclusion "Sobolev is inert" needs reframing, and the analysis has a
   factor-2 error.
3. The reason recorded for retiring the M⁸ comparison is backwards; in 2D the law is M⁷.
4. Several numbers and arguments in the `branch_analysis.md` plan are wrong.

**Labels used below:** **[computed]** recomputed in this notebook · **[cited]** quoted
from a source · **[code]** read from the repository · **[unverified]** not confirmed.

**Inputs:** `turb2d.npz`. **Runtime:** ~10 min.

In [ ]:
# Setup -- run this cell first.
# In Colab: clones the private repo (needs the Colab secret GH_TOKEN) and mounts Google
# Drive, where the datasets live in MyDrive/silent-field/data/.  Locally: finds the repo.
import subprocess, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    _repo = Path("/content/silent-field")
    if not (_repo / ".git").exists():
        from google.colab import userdata
        _url = f"https://{userdata.get('GH_TOKEN')}@github.com/HSajeed/silent-field.git"
        subprocess.run(["git", "clone", "-q", "--branch", "tier0-review-notebooks", _url, str(_repo)],
                       check=True)
        subprocess.run(["git", "-C", str(_repo), "remote", "set-url", "origin",
                        "https://github.com/HSajeed/silent-field.git"], check=True)
    sys.path.insert(0, str(_repo / "notebooks"))
except ImportError:
    _here = Path.cwd()
    sys.path.insert(0, str(_here if (_here / "nbtools").exists() else _here / "notebooks"))

from nbtools import env as nbenv, style
from nbtools.style import Comparison, md

ENV = nbenv.setup()
style.use()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from nbtools import spectra, cache
from src.acoustics.komega import sound_speed, komega_spectrum, KOmegaSpectrum
from src.acoustics.lighthill import source_timeseries
from src.acoustics.greens import source_spectrum, far_field_pressure, far_field_pressure_ibp, observer_ring

nbenv.verify_dataset(ENV, "turb2d")
u, v, grid, dt = spectra.load(ENV, "turb2d")
U = float(np.sqrt(np.mean(u**2 + v**2)))
c0 = sound_speed(U, 0.1)
L = grid.length
print(f"u_rms = {U:.4f}, c0 (M = 0.1) = {c0:.3f}, first box mode dk = {2 * np.pi / L:.3f}")

## 1. The far field uses the wrong form of the source for a periodic box

### 1.1 What the literature says [cited]
Sarkar & Hussaini (1993), *Computation of the sound generated by isotropic turbulence*,
NASA CR-191543 / ICASE 93-74, is the standard reference for Lighthill sound from turbulence
in a periodic box. They compare two ways of writing the far field:
- **space-derivative form (their Eq. 1)**: radiate $S=\partial_i\partial_jT_{ij}$;
- **time-derivative form (their Eq. 2)**: radiate $T_{ij}$ itself, with both spatial
  derivatives moved onto the Green's function.

Their verdict, verbatim:

> "Eq. (1) is not quite appropriate here because of the periodic boundary conditions used
> for the turbulence simulation. Because of the non-zero velocity at the fluid boundary of
> the periodic box, the dipole contribution to the sound from the surface of the flow
> domain is non-zero and would be included in a calculation using Eq. (1). The surface
> contribution is an artifact of the boundary conditions and is not of interest to us.
> Eq. (2), on the other hand, gives directly the quadrupole sound characteristic of a
> finite volume of turbulent flow embedded in a medium at rest."

### 1.2 What the code does [code]
- `scripts/03_compute_sources.py` radiates S with `far_field_pressure`: the space-derivative
  form ("**S-form**").
- R_S (chapter 03) is the cone share of S.
- `far_field_pressure_ibp` (`src/acoustics/greens.py:270`) integrates $\partial_i\partial_jG$
  against $T_{ij}$ ("**T-form**"). It was written to avoid differentiating a noisy surrogate
  twice, and is not called anywhere. Chapter 01 §6 showed it reproduces the S-form on the
  vortex pair at its tone.

### 1.3 What S cannot contain [computed]

On a periodic grid the k = 0 Fourier mode of a derivative is exactly zero, so the box
mean of S vanishes. The box integral of $T_{ij}=u_iu_j$ does not. Its fluctuation in time
is the *compact quadrupole*: the dominant radiation of a turbulent region small compared
with the acoustic wavelength.

In [ ]:
S64 = source_timeseries(u[:64], v[:64], grid)
print(f"max |box mean of S| over 64 frames: {np.abs(S64.mean(axis=(1, 2))).max():.1e}   (rms of S: {S64.std():.1f})")
for name, T in (("T_xx", u * u), ("T_xy", u * v), ("T_yy", v * v)):
    Tm = T.mean(axis=(1, 2))
    print(f"box-integrated {name}: mean {Tm.mean():+.4f}, fluctuation std {Tm.std():.4f}")
del S64

### 1.4 Both far fields from the same truth data [computed]

M = 0.1, 36 observers at r = 30L, full-record Hann. "Compact share" is the T-form with
$T_{ij}$ replaced by its box mean (k = 0 content only), divided by the full T-form.

In [ ]:
def build_far_field():
    S = source_timeseries(u, v, grid)
    sh, om = source_spectrum(S, dt); del S
    th = {key: source_spectrum(a, dt)[0] for key, a in (("uu", u * u), ("uv", u * v), ("vv", v * v))}
    obs = observer_ring(30 * L, 36, (L / 2, L / 2))
    idx = sorted({int(np.argmin(abs(om - w))) for w in (1.5, 3, 6, 9, 11.3, 15, 25, 40, 60)})
    pS = far_field_pressure(sh, om, grid, obs, c0, True, idx)
    pT = far_field_pressure_ibp(th, om, grid, obs, c0, idx)
    thm = {key: np.broadcast_to(a.mean(axis=(1, 2), keepdims=True), a.shape) for key, a in th.items()}
    pC = far_field_pressure_ibp(thm, om, grid, obs, c0, idx)
    rms = lambda p: np.sqrt(np.mean(np.abs(p) ** 2, axis=1))
    return {"omega": om[idx], "S": rms(pS), "T": rms(pT), "C": rms(pC)}

ff = cache.array(ENV.cache, "turb2d_far_field_S_vs_T", build_far_field)
ratio = ff["S"] / ff["T"]
md("| ω | k_ac / dk | \\|p\\| S-form (used) | \\|p\\| T-form | **S / T** | compact share of T |\n|---|---|---|---|---|---|\n" +
   "".join(f"| {w:.2f} | {w / c0 / (2 * np.pi / L):.2f} | {s:.2e} | {t:.2e} | **{r:.0f}** | {c / t:.2f} |\n"
           for w, s, t, r, c in zip(ff["omega"], ff["S"], ff["T"], ratio, ff["C"])))
REVIEW = [168.04, 114.87, 54.66, 20.35, 20.25, 20.04, 15.72, 16.39, 6.98]
assert np.allclose(ratio, REVIEW, rtol=2e-3), "does not reproduce review_2026-10-01.md §1.3"
plt.loglog(ff["omega"], ff["S"], "o-", color=style.OLD, label="S-form (used by scripts/03)")
plt.loglog(ff["omega"], ff["T"], "s-", color=style.NEW, label="T-form (Sarkar & Hussaini Eq. 2)")
plt.axvline(c0 * 2 * np.pi / L, color=style.REF, ls=":", label=r"$c_0\,dk$ (Tier-0 band starts here)")
plt.xlabel(r"$\omega$"); plt.ylabel("rms |p| on the observer ring"); plt.legend(); plt.show()

**Reading:**
- **The S-form far field is 7–168× larger than the T-form at every frequency,** including
  16–20× inside the band Tier 0 selected. A Stage-3 far-field error computed with the
  S-form would mostly compare the boundary artifacts of truth and prediction.
- **The excess falls roughly as 1/k at low ω, then levels off.** A surface dipole (∝ k)
  set against a volume quadrupole (∝ k²) would produce that pattern. It is consistent with
  Sarkar & Hussaini's diagnosis, though not a proof of it.
- **Below the first box mode, the T-form radiation is mostly the compact term** (68–97%).
  That is exactly what the S-based cone share cannot see, because S has no k = 0 content.
- **This is the likely reason R depends on the box (chapter 03):** a bigger box moves
  radiation from the unrepresentable gap (0, dk) onto grid modes. The box itself is
  ~12× the eddy size (chapter 02), which meets Sarkar & Hussaini's size requirement.

**Consequences:**
- switch `scripts/03` to `far_field_pressure_ibp`;
- revisit the Tier-0 band, since frequencies below c₀·dk do radiate in the T-form;
- re-define R_S on $T_{ij}$;
- add the S = T check on the vortex pair as a gate check (chapter 01 §6).

**Still to do** **[unverified]**: derive and test the 2D cone metric on $T_{ij}$, roughly
$\propto k_{ac}^4\,|n_in_j\hat T_{ij}(k_{ac}\hat n,\omega)|^2$, before it replaces R_S.

## 2. Stage 4: "Sobolev is inert" needs reframing

### 2.1 At the default λ the Sobolev term is negligible everywhere [computed]

The trained objective's per-mode weight is $w=1+\lambda\,c\,\tilde w(k)$, where
$\tilde w$ is (1+k²) normalised to mean 1 and c ≈ 1 per full-plane mode (the current code,
after the factor-2 fix of 2026-10-07, §2.2; at the review it was ≈ 2). The share of the
whole objective coming from the Sobolev term is $\sum P(w-1)/\sum Pw$.

In [ ]:
from src.acoustics.radiating import additive_sobolev_weight, shell_axis_fraction, sobolev_halfplane_mean_weight
spec = spectra.spectra_for(ENV, "turb2d", source=False)["vel_full"]
P = spec.power_resolved.sum(axis=0)                     # energy per resolved |k|
shell = np.rint(grid.k_sq.ravel() / (2 * np.pi / L) ** 2).astype(int)
modes_per_k = np.bincount(shell)[np.unique(shell)].astype(float)
axis_frac = shell_axis_fraction(spec.k_resolved, grid)
for lam in (1e-2, 1.0, 1e2):
    w = additive_sobolev_weight(spec.k_resolved, grid, lam, axis_fraction=axis_frac)
    frac_data = (P * (w - 1)).sum() / (P * w).sum()
    frac_white = (modes_per_k * (w - 1)).sum() / (modes_per_k * w).sum()
    print(f"lambda = {lam:g}: Sobolev term is {frac_data:.2%} of the objective for error shaped like the data, "
          f"{frac_white:.1%} for white error")

So at λ = 1e-2 the trained "Sobolev" objective *is* L2 to within the percentage printed
above (0.03% at the review, before the factor-2 fix of §2.2; 0.01% now),
everywhere, not just in the cone. "Inert at λ = 1e-2" mostly reflects how small λ is. A referee will read
it as "Sobolev training was switched off".

**The robust statement:**
- $w\ge1$ everywhere and $w\approx1$ in the cone, so Sobolev weighting can **never raise**
  the cone share;
- wherever it is strong enough to change training, it **lowers** it, by up to ~19×
  (λ sweep, chapter 04).

That holds for any error spectrum and any λ.

### 2.2 The analysis weight was 2× too large for most modes [computed; fixed 2026-10-07]

At the review, `additive_sobolev_weight` used r = n/(n//2+1) per mode
(`src/acoustics/radiating.py:119`). Each interior rfft2 coefficient stands for two
full-plane modes (+k and −k), so per mode the correct factor is n/(n+2), half as large,
for every mode with kx ≠ 0; only modes on the kx = 0 (and Nyquist) columns get n/(n//2+1).
Measured against the real torch loss on single Fourier modes, with the formula as it was
at the review written out inline and the current function (axis fraction 1 for a kx = 0
mode, 0 otherwise):

In [ ]:
from src.models.losses import sobolev_loss
from src.spectral import Grid
g64, lam = Grid(n=64, length=2 * np.pi), 0.37
x = np.arange(64) * g64.length / 64
X, Y = np.meshgrid(x, x, indexing="xy")
wbar = sobolev_halfplane_mean_weight(g64)
rows = []
for mx, my in [(3, 5), (0, 4), (7, 0)]:
    d = np.cos(mx * X + my * Y); dt_ = torch.tensor(d)[None, None]
    measured = sobolev_loss(dt_, torch.zeros_like(dt_), 1, lam, g64.length).item() / np.mean(d**2) - 1
    k_mode = np.hypot(mx, my)
    at_review = lam * (g64.n / (g64.n // 2 + 1)) * (1 + k_mode**2) / wbar
    current = additive_sobolev_weight(np.array([k_mode]), g64, lam, axis_fraction=float(mx == 0))[0] - 1
    rows.append(((mx, my), measured, at_review, current, at_review / measured, current / measured))
md("| mode (kx, ky) | w − 1, torch loss | w − 1, formula at review | w − 1, current code | review / loss | current / loss |\n|---|---|---|---|---|---|\n" +
   "".join(f"| {m} | {a:.4e} | {b:.4e} | {c:.4e} | **{d:.3f}** | **{e:.3f}** |\n" for m, a, b, c, d, e in rows))
assert np.allclose([r[4] for r in rows], [2, 1, 2], atol=1e-6)
assert np.allclose([r[5] for r in rows], [1, 1, 1], atol=1e-6)

- **Why the tests missed it** (at the review): `tests/test_radiating.py:100` rebuilt the same ratio rather
  than evaluating the loss, so it was circular.
- **Direction of the error:** it overstated the Sobolev effect, so no conclusion flips;
  the λ axis of the sweep shifts by 2×.
- **A side effect in the loss itself:** modes on the kx = 0 column get twice the Sobolev
  weight of other modes at the same |k|, because the half-plane mean in `losses.py` has no
  Hermitian weights.

### 2.3 The standard practitioner loss is the multiplicative one [cited]
neuraloperator's `H1Loss` (`neuralop/losses/data_losses.py`) sums squared differences of
the field and of its derivatives with unit weight:

```python
diff = torch.sum((dict_x[0] - dict_y[0]) ** 2, ...)
for j in range(1, self.d + 1):
    diff += torch.sum((dict_x[j] - dict_y[j])**2, ...)
```

That is ‖e‖² + ‖∇e‖², i.e. (1 + k²) up to finite-difference error: the *multiplicative*
reading (0.053× L2 here). Use that form for trained Stage-4 runs.

Also [cited]: Czarnecki et al. 2017 (arXiv:1706.04859) defines Sobolev training as matching
"target derivatives in addition to the target values", i.e. derivatives with respect to the
network's inputs. It is not a spectral field loss, so cite H¹ losses as used with neural
operators instead.

## 3. Mach scaling: retire M⁸ for the right reason

**The decision log says** (HANDOFF): M⁸ requires a compact source, compactness is
ωL/c₀ ~ M, "so the low-Mach limit is precisely where it breaks".

**That is backwards** [cited]. Schram (POLKA lecture, p. 22): "Acoustical compactness
expressed by Helmholtz number: He = 2fD/c₀ … Acoustical compactness depends on Mach number:
He = 2Sr U/c₀ = 2Sr M". A source is compact when He ≪ 1, and He ∝ M, so **compactness
improves as M → 0**.

In [ ]:
Sr = 0.2   # a typical Strouhal number
md("| M | He = 2·Sr·M (Sr = 0.2) |\n|---|---|\n" +
   "".join(f"| {M} | {2 * Sr * M:.2f} |\n" for M in (0.05, 0.1, 0.2, 0.3, 0.6)) +
   "\nHe falls in proportion to M: the lower the Mach number, the more compact the source.")

- **In 2D the benchmark is M⁷, not M⁸** [cited]. Schoder et al. (arXiv:2608.19236): "the
  M⁷ power law expected for compact quadrupole sources in two dimensions (Ffowcs Williams
  1969)". M⁸ is the 3D law.
- **Steep exponents alone do not prove an artifact** [cited]. The same paper finds 2D
  mixing-layer power "grows more steeply than the M⁷ law" at M = 0.2–0.4 (with Reynolds
  number varying alongside Mach).
- **What to do:** compare a *radiated power* computed with the T-form (§1) against M⁷. R is
  an energy share, not a power, and should not be compared with either law.

## 4. The `branch_analysis.md` plan: what needs correcting

### 4.1 Its §3 table of error models [computed]
`branch_analysis.md` proposes the **blindness ratio** B = R_loss / R_signal: the cone share
of the *error* over the cone share of the *signal*. That is the right idea. But its table of
example error shapes has two wrong rows. Recomputed on the same estimator it used (single
full-record Hann, exact mask):

In [ ]:
def hann_full():
    su = komega_spectrum(u, grid, dt); sv = komega_spectrum(v, grid, dt)
    return KOmegaSpectrum(power=su.power + sv.power, k_bins=su.k_bins, omega=su.omega,
                          k_resolved=su.k_resolved, power_resolved=su.power_resolved + sv.power_resolved)

sh1 = cache.spectrum(ENV.cache, "turb2d_vel_hann_full", KOmegaSpectrum, hann_full)
Pk, kr, om = sh1.power_resolved, sh1.k_resolved, sh1.omega
mask = sh1.radiating_mask_resolved(c0)
share = lambda W: float((W * mask).sum() / W.sum())
white = np.broadcast_to(modes_per_k[None, :], Pk.shape).astype(float)
Comparison("cone share of the loss for three error shapes, M = 0.1").add(
    "error ∝ signal (what R assumes)", new=share(Pk), logged=5.18e-08, check="new", rtol=2e-3, source="review §4.2").add(
    "white error (branch_analysis said 6.2e-2)", new=share(white), logged=3.03e-03, check="new", rtol=2e-3,
    source="review §4.2").add(
    "error ∝ k²·signal (branch_analysis: 'higher still')", new=share(Pk * kr[None, :] ** 4), logged=8.82e-12,
    check="new", rtol=2e-3, source="review §4.2").show()
print(f"(this estimator puts {Pk[:, np.isclose(kr, 1)].sum() / Pk.sum():.1%} of the energy in the |k| = 1 shell; "
      "the Welch estimate in chapter 03 puts 2.2% there)")

- **White error: 3.0e-3, not 6.2e-2.** 6.2e-2 is the cone's *area* in the (k, ω) plane. A
  white error has equal power per *grid mode*, and the cone holds few modes.
- **k²-shaped error: 8.8e-12, not "higher".** Weighting by k⁴ pushes the error to high k,
  away from the low-k cone.
- **Unchanged by these corrections:** a trained model's cone loss share can be orders of
  magnitude above R, which is why B is worth measuring (chapter 06).

### 4.2 How to read B
- **"B ≥ 1" is not a theorem.** A model can be relatively *better* inside the cone than
  outside it. Example: an error that is 1% of the signal outside the cone and 0.1% inside
  gives B = 0.01.
- **The training loss has no frequency axis** [code]. `make_pairs` builds one-step
  (snapshot → next snapshot) pairs, and the loss compares single snapshots. It cannot
  favour or neglect high ω. The cone at M = 0.1 is the high-ω tail of the |k| = 1 shell.
- **Hypothesis to test** [unverified]: step-to-step incoherence (jitter) in teacher-forced
  predictions puts error into the cone regardless of any energy weighting. Stage 3 needs
  two controls: (i) teacher-forced vs free rollout, (ii) a random-error baseline at
  matched amplitude.
- **Measure B on the source error too (B_S), using the T-form.** The source is quadratic in
  velocity, so its radiating content comes from pairs of energetic velocity modes that are
  not themselves in the cone.

In [ ]:
from src.models.train import make_pairs
x, y, _ = make_pairs(u[:5], v[:5])
print(f"one-step pairs: input {tuple(x.shape)} -> target {tuple(y.shape)}  (batch, components, ny, nx): "
      "no time axis inside a training example")

### 4.3 Other items in the plan

| item | verdict | evidence |
|---|---|---|
| §6F "a strictly periodic source has no far field (Rayleigh's theorem)" | ❌ unsupported | Periodic arrays radiate through propagating Rayleigh–Bloch orders \|κₙ\| ≤ ω (Barnett & Greengard, quasi-periodic scattering in 2D) [cited]. The real problem is §1 above. |
| §6G lower the drag to get k⁻³ | ❌ risky | Condensate at the box scale (Boffetta & Ecke 2012 §4.2; Chertkov et al. 2007) [cited]; chapter 02 |
| §4 "low Mach is where compactness breaks" | ❌ backwards | §3 above |
| §6B "λ-independent where Sobolev operates" | ❌ backwards | §2.1 above |
| §7 "do not implement a cone-weighted loss" | ⚠️ partly | A causal intervention strengthens a diagnosis. The cheap one is a multi-step / temporal loss (chapter 07). |
| Rucz et al. 2024 as the new gate reference | ✅ | J. Sound Vib. 587, 118496 [cited] |
| Duraisamy, arXiv:2604.20061 | ✅ exists | abstract confirms the spectral-bias framing; "via NTK" [unverified] |
| Kolster et al. / NASA-CR-208109 | ❓ | could not be located [unverified] |

### 4.4 A small documentation nit [computed]
HANDOFF says the resolved grid has "~1850 values at n = 128", and `komega.py:139` says the
k² values are "exact integers", so modes "provably" share a group. In fact `grid.k_sq`
deviates from integers by ~1e-12, so `np.unique` splits some shells:

In [ ]:
ks = grid.k_sq.ravel()
print(f"distinct float k^2: {np.unique(ks).size};  distinct integer k^2: {np.unique(np.rint(ks)).size};  "
      f"max deviation from an integer: {np.abs(ks - np.rint(ks)).max():.1e}")
print("harmless for the mask (split groups have the same |k| to 1e-12), but the docstring was wrong")
print("since 2026-10-07 `komega.py` groups on rint(k^2/dk^2): 1621 resolved shells at n = 128 (1850 before); metric values unchanged to 6e-16")

## What this chapter established
- **The S-form far field is dominated by an artifact** (7–168×). The T-form, already in
  the code, is the published remedy. This is now the top item of the plan.
- **Sobolev:** "can never raise the cone share, and lowers it wherever it matters" replaces
  "inert". The analysis weight needs its factor 2 fixed.
- **Mach scaling:** M⁸ is retired because it is the 3D law, not because compactness
  fails. The 2D benchmark is M⁷.
- **`branch_analysis.md`:** two of its three error-model numbers were wrong. B is still
  the right idea, but "B ≥ 1" is not a theorem, and the one-step loss's blindness to
  frequency must be tested with controls.